# Build PCA Tables

This notebook applies a PCA workflow to the constitution project by using the reduced and L2-normalized TFIDF matrix from `tfidf_reduced_l2.parquet`. It follows the general HW7 pattern while adapting the outputs to the project tables, metadata, and the PCA reporting fields in `FinalProject.ipynb`.

## Inputs and Outputs

**Input files**
- `tfidf_reduced_l2.parquet`
- `lib.csv`

**Output files**
- `pca_comps.parquet`: component-by-term table in which each row is a principal component and each column is a vocabulary term
- `pca_dcm.parquet`: document-component matrix in which each row is a document bag and each column is a principal component score
- `pca_loadings.parquet`: term-by-component table used to inspect the strongest positive and negative terms for each component
- `pca_explained_variance.parquet`: explained variance statistics for each principal component

**Visualization outputs**
- constitution-level scatterplots for `PC0` vs `PC1`
- constitution-level scatterplots for `PC2` vs `PC3`
- matching loading-space scatterplots for the same component pairs
- a scree plot for explained variance

The notebook also prints project-ready PCA summary details in notebook cells so the final project notebook can be updated directly without relying on a separate summary CSV.


In [75]:
# Maintainer note: this PCA workflow is adapted from the MO7 / `nbz3de-m07-hw.ipynb` component-analysis pattern.
# Keep the homework provenance here in code comments instead of the user-facing project documentation.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = 'pyarrow'
except ModuleNotFoundError:
    try:
        import fastparquet  # noqa: F401
        PARQUET_ENGINE = 'fastparquet'
    except ModuleNotFoundError:
        PARQUET_ENGINE = None

try:
    import plotly.express as px
    import plotly.io as pio
    # Use a renderer that works better in VS Code and modern notebook frontends.
    pio.renderers.default = 'plotly_mimetype'
except ModuleNotFoundError:
    px = None
    pio = None

PCA_INPUT_PARQUET = Path('tfidf_reduced_l2.parquet')
LIB_CSV = Path('lib.csv')

COMPS_PARQUET = Path('pca_comps.parquet')
DCM_PARQUET = Path('pca_dcm.parquet')
LOADINGS_PARQUET = Path('pca_loadings.parquet')
EXPLAINED_VARIANCE_PARQUET = Path('pca_explained_variance.parquet')

print(f'Parquet engine: {PARQUET_ENGINE or "unavailable"}')


Parquet engine: fastparquet


## Load Inputs

In [76]:
# Load the reduced TFIDF matrix and reconstruct the intended OHCO bag index.
TFIDF_REDUCED_L2_RAW = pd.read_parquet(PCA_INPUT_PARQUET)
PCA_INPUT_USED = PCA_INPUT_PARQUET
TFIDF_REDUCED_L2_RESET = TFIDF_REDUCED_L2_RAW.reset_index()
INDEX_COLS = [col for col in ['country_id', 'article_n', 'clause_n'] if col in TFIDF_REDUCED_L2_RESET.columns]
if INDEX_COLS == ['country_id']:
    PCA_BAG_LEVEL = 'DOCS'
elif INDEX_COLS == ['country_id', 'article_n']:
    PCA_BAG_LEVEL = 'ARTICLES'
elif INDEX_COLS == ['country_id', 'article_n', 'clause_n']:
    PCA_BAG_LEVEL = 'CLAUSES'
else:
    PCA_BAG_LEVEL = 'CUSTOM'
TFIDF_REDUCED_L2 = TFIDF_REDUCED_L2_RESET.set_index(INDEX_COLS)

# Load document metadata that can be joined onto PCA document scores.
LIB = pd.read_csv(LIB_CSV)

print(f'TFIDF_REDUCED_L2 shape: {TFIDF_REDUCED_L2.shape}')
print(f'PCA bag level: {PCA_BAG_LEVEL} -> {INDEX_COLS}')
print(f'LIB shape: {LIB.shape}')
if 'country_id' in TFIDF_REDUCED_L2.index.names:
    countries_in_matrix = TFIDF_REDUCED_L2.index.get_level_values('country_id').nunique()
else:
    countries_in_matrix = TFIDF_REDUCED_L2.index.nunique()
print(f'Countries in PCA matrix: {countries_in_matrix}')
TFIDF_REDUCED_L2.iloc[:5, :5]

TFIDF_REDUCED_L2 shape: (33726, 3006)
PCA bag level: ARTICLES -> ['country_id', 'article_n']
LIB shape: (192, 19)
Countries in PCA matrix: 192


term_str               abide  ability  able  abolish  abolished
country_id  article_n                                          
Afghanistan 1            0.0      0.0   0.0      0.0        0.0
            2            0.0      0.0   0.0      0.0        0.0
            3            0.0      0.0   0.0      0.0        0.0
            4            0.0      0.0   0.0      0.0        0.0
            5            0.0      0.0   0.0      0.0        0.0

## Build PCA

The notebook uses the rows of `TFIDF_REDUCED_L2` as documents and the columns as terms. If the reduced TFIDF table is article- or clause-level, PCA is fit on those finer-grained bags. Later visualization cells aggregate the resulting PCA scores back to one row per constitution for readable constitution-level plots.

This section creates the four core PCA outputs required by the final project:
- `build_pca_model()` fits the PCA estimator on the reduced L2-normalized TFIDF matrix.
- `build_pca_comps()` creates `pca_comps.parquet`, where rows are principal components and columns are term weights.
- `build_pca_dcm()` creates `pca_dcm.parquet`, where rows are document bags expressed in OHCO levels and columns are principal component scores.
- `build_pca_loadings()` creates `pca_loadings.parquet`, where rows are terms and columns are principal components for interpretation.
- `build_explained_variance()` creates `pca_explained_variance.parquet`, which stores explained variance and cumulative explained variance for each component.


In [77]:
def build_pca_model(tfidf_l2: pd.DataFrame, n_components: int | None = None) -> PCA:
    # Fit PCA on the reduced L2-normalized TFIDF matrix.
    pca = PCA(n_components=n_components)
    pca.fit(tfidf_l2)
    return pca


def build_pca_comps(pca: PCA, terms: pd.Index) -> pd.DataFrame:
    # Store each principal component as a row of term weights.
    pc_names = [f'PC{i}' for i in range(len(pca.components_))]
    comps = pd.DataFrame(pca.components_, index=pc_names, columns=terms)
    comps.index.name = 'component_id'
    return comps


def build_pca_dcm(pca: PCA, tfidf_l2: pd.DataFrame) -> pd.DataFrame:
    # Project each country_id onto the PCA component axes.
    pc_names = [f'PC{i}' for i in range(len(pca.components_))]
    dcm = pd.DataFrame(pca.transform(tfidf_l2), index=tfidf_l2.index, columns=pc_names)
    dcm.index.name = 'country_id'
    return dcm


def build_pca_loadings(comps: pd.DataFrame) -> pd.DataFrame:
    # Transpose component weights so terms can be inspected by component.
    loadings = comps.T.copy()
    loadings.index.name = 'term_str'
    return loadings


def build_explained_variance(pca: PCA) -> pd.DataFrame:
    # Capture explained variance statistics for reporting and interpretation.
    pc_names = [f'PC{i}' for i in range(len(pca.components_))]
    explained = pd.DataFrame({
        'component_id': pc_names,
        'explained_variance': pca.explained_variance_,
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'cumulative_explained_variance_ratio': np.cumsum(pca.explained_variance_ratio_),
    })
    return explained


PCA_MODEL = build_pca_model(TFIDF_REDUCED_L2)
COMPS = build_pca_comps(PCA_MODEL, TFIDF_REDUCED_L2.columns)
DCM = build_pca_dcm(PCA_MODEL, TFIDF_REDUCED_L2)
LOADINGS = build_pca_loadings(COMPS)
EXPLAINED = build_explained_variance(PCA_MODEL)

print('COMPS shape:', COMPS.shape)
print('DCM shape:', DCM.shape)
print('LOADINGS shape:', LOADINGS.shape)
print('EXPLAINED shape:', EXPLAINED.shape)
EXPLAINED.head(10)

COMPS shape: (3006, 3006)
DCM shape: (33726, 3006)
LOADINGS shape: (3006, 3006)
EXPLAINED shape: (3006, 4)


,component_id,explained_variance,explained_variance_ratio,cumulative_explained_variance_ratio
0,PC0,0.006741,0.006832,0.006832
1,PC1,0.004989,0.005057,0.011889
2,PC2,0.004410,0.004470,0.016359
3,PC3,0.004127,0.004184,0.020542
4,PC4,0.003723,0.003773,0.024315
5,PC5,0.003652,0.003702,0.028017
6,PC6,0.003397,0.003443,0.031460
7,PC7,0.003265,0.003310,0.034770
8,PC8,0.003211,0.003254,0.038024
9,PC9,0.003198,0.003241,0.041266


## Join Metadata and Visualize

This step mirrors the HW7 interpretation pattern by joining PCA scores to metadata from `LIB`. When the PCA input is article- or clause-level, the notebook preserves those bag-level scores but aggregates them back to one row per constitution for the main country_id-level visualizations.

In [78]:
def build_pca_bag_table(dcm: pd.DataFrame, lib: pd.DataFrame) -> pd.DataFrame:
    # Join country_id-level metadata onto whichever bag level was used to fit PCA.
    lib_doc = lib.drop_duplicates(subset=['country_id'])
    dcm_doc = dcm.reset_index()
    merged = dcm_doc.merge(lib_doc, on='country_id', how='left')
    index_cols = [col for col in ['country_id', 'article_n', 'clause_n'] if col in merged.columns]
    return merged.set_index(index_cols)


def build_pca_country_table(dcm: pd.DataFrame, lib: pd.DataFrame) -> pd.DataFrame:
    # Aggregate bag-level PCA scores back to one row per constitution for readable plots.
    pc_cols = [col for col in dcm.columns if str(col).startswith('PC')]
    dcm_reset = dcm.reset_index()
    country_scores = dcm_reset.groupby('country_id')[pc_cols].mean()
    country_scores['n_bags'] = dcm_reset.groupby('country_id').size()
    lib_doc = lib.drop_duplicates(subset=['country_id']).set_index('country_id')
    return country_scores.join(lib_doc, how='left')


def vis_pcs(data: pd.DataFrame, a: int, b: int, color_col: str, hover_col: str = 'country_id'):
    # Plot countries in the space defined by two principal components.
    if px is None:
        raise ModuleNotFoundError('plotly is required for vis_pcs() but is not installed in this environment.')
    plot_data = data.reset_index().copy()
    legend_labels = {
        'region_compressed': 'Region',
        'year_created': 'Year Created',
        'year_amended': 'Year Amended',
        'v2x_regime_cat': 'V-Dem Regime Category',
        'v2x_freexp_altinf_cat': 'V-Dem Freedom of Expression Category',
        'v2x_rule_cat': 'V-Dem Rule of Law Category',
    }
    value_maps = {
        'v2x_freexp_altinf_cat': {
            'Very low': 'Very Low Freedom of Expression',
            'Low': 'Low Freedom of Expression',
            'Moderate': 'Moderate Freedom of Expression',
            'High': 'High Freedom of Expression',
            'Very high': 'Very High Freedom of Expression',
            'Unknown': 'Freedom of Expression Unknown',
        },
        'v2x_rule_cat': {
            'Very low': 'Very Low Rule of Law',
            'Low': 'Low Rule of Law',
            'Moderate': 'Moderate Rule of Law',
            'High': 'High Rule of Law',
            'Very high': 'Very High Rule of Law',
            'Unknown': 'Rule of Law Unknown',
        },
    }
    if color_col in value_maps:
        plot_data[color_col] = plot_data[color_col].replace(value_maps[color_col])
    return px.scatter(
        plot_data,
        x=f'PC{a}',
        y=f'PC{b}',
        color=color_col,
        hover_name=hover_col,
        marginal_x='box',
        height=800,
        labels={
            f'PC{a}': f'PC{a}',
            f'PC{b}': f'PC{b}',
            color_col: legend_labels.get(color_col, color_col),
        },
    )


def vis_loadings(loadings: pd.DataFrame, a: int = 0, b: int = 1, top_n: int = 0):
    # Plot term loadings in the space defined by two principal components.
    if px is None:
        raise ModuleNotFoundError('plotly is required for vis_loadings() but is not installed in this environment.')
    data = loadings.reset_index().copy()
    if top_n > 0:
        # Keep the most extreme terms across the two plotted components for readability.
        score = data[[f'PC{a}', f'PC{b}']].abs().max(axis=1)
        data = data.loc[score.nlargest(top_n).index].copy()
    return px.scatter(
        data,
        x=f'PC{a}',
        y=f'PC{b}',
        text='term_str',
        hover_name='term_str',
        marginal_x='box',
        height=800,
    )


def vis_scree(explained: pd.DataFrame, top_n: int = 10):
    # Plot explained variance ratios for the leading principal components.
    if px is None:
        raise ModuleNotFoundError('plotly is required for vis_scree() but is not installed in this environment.')
    return px.bar(
        explained.head(top_n),
        x='component_id',
        y='explained_variance_ratio',
        title='Explained Variance Ratio by Principal Component',
    )


PCA_BAG = build_pca_bag_table(DCM, LIB)
PCA_COUNTRY = build_pca_country_table(DCM, LIB)

print(f'Bag-level PCA table shape: {PCA_BAG.shape}')
print(f'Country-level PCA table shape: {PCA_COUNTRY.shape}')
PCA_COUNTRY.head()


Bag-level PCA table shape: (33726, 3024)
Country-level PCA table shape: (192, 3025)


,PC0,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,line_count,article_count,clause_count,token_count,v2x_regime,v2x_freexp_altinf,v2x_rule,v2x_regime_cat,v2x_freexp_altinf_cat,v2x_rule_cat
country_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan,0.026638,-0.017579,0.009407,0.001150,-0.020391,-0.004640,0.016732,0.009368,0.008699,-0.014779,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
Albania,0.012192,-0.005823,0.018732,0.007241,-0.024900,0.019407,-0.003408,-0.000461,0.005247,0.007233,...,3327,183,582,13221,2.0,0.807,0.516,Electoral democracy,Very high,Moderate
Algeria,0.026296,-0.013678,-0.002481,-0.007505,-0.025608,0.003481,0.006942,0.014041,0.002554,-0.012927,...,2098,185,460,10072,1.0,0.662,0.250,Electoral autocracy,High,Low
Andorra,0.018075,-0.002018,0.022618,-0.025166,-0.009397,0.000187,-0.015748,-0.003493,-0.015122,-0.000689,...,1668,108,277,8310,NaN,NaN,NaN,Unknown,Unknown,Unknown
Angola,0.050340,0.002382,0.019396,0.002082,-0.004923,-0.013174,0.002587,-0.004238,-0.010693,0.011515,...,5274,250,1123,24586,1.0,0.361,0.129,Electoral autocracy,Low,Very low


## Inspect Component Terms

Use the helper below to read the strongest positive and negative term weights for any component when writing the PCA interpretation sections in the final project notebook.

In [79]:
def top_component_terms(loadings: pd.DataFrame, component_id: str, top_n: int = 10) -> tuple[pd.Series, pd.Series]:
    # Return the strongest positive and negative terms for one component.
    weights = loadings[component_id].sort_values(ascending=False)
    top_pos = weights.head(top_n)
    top_neg = weights.tail(top_n).sort_values()
    return top_pos, top_neg


pc0_pos, pc0_neg = top_component_terms(LOADINGS, 'PC0', top_n=10)
print('Top positive PC0 terms:')
print(pc0_pos)
print('\nTop negative PC0 terms:')
print(pc0_neg)

Top positive PC0 terms:
term_str
social         0.189188
economic       0.139364
citizens       0.122873
development    0.122216
protection     0.119799
education      0.119240
freedom        0.114748
people         0.097697
property       0.094303
human          0.091064
Name: PC0, dtype: float64

Top negative PC0 terms:
term_str
speaker            -0.178459
deputy             -0.122721
ministers          -0.118989
majority           -0.109346
senate             -0.108589
governor-general   -0.107411
judge              -0.104304
vote               -0.099787
first              -0.095581
session            -0.095354
Name: PC0, dtype: float64


## Final Project Summary

This section computes compact summaries that map directly to the PCA fields in `FinalProject.ipynb`, including the number of components, top terms, explained variance, bag level, and table metadata for `pca_comps.parquet`, `pca_dcm.parquet`, `pca_loadings.parquet`, and `pca_explained_variance.parquet`.


In [80]:
def build_component_summary(loadings: pd.DataFrame, explained: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    # Gather top positive and negative terms for each component alongside explained variance.
    rows = []
    explained_index = explained.set_index('component_id')
    for component_id in explained['component_id']:
        top_pos, top_neg = top_component_terms(loadings, component_id, top_n=top_n)
        rows.append({
            'component_id': component_id,
            'explained_variance': explained_index.loc[component_id, 'explained_variance'],
            'explained_variance_ratio': explained_index.loc[component_id, 'explained_variance_ratio'],
            'cumulative_explained_variance_ratio': explained_index.loc[component_id, 'cumulative_explained_variance_ratio'],
            'top_positive_terms': ', '.join(top_pos.index.tolist()),
            'top_negative_terms': ', '.join(top_neg.index.tolist()),
        })
    return pd.DataFrame(rows)


def build_project_info(comps: pd.DataFrame, dcm: pd.DataFrame, loadings: pd.DataFrame, explained: pd.DataFrame) -> pd.DataFrame:
    # Collect one-row metadata fields that can be copied into the final project notebook.
    explained_index = explained.set_index('component_id')
    pc0_pos, pc0_neg = top_component_terms(loadings, 'PC0', top_n=5)
    pc1_pos, pc1_neg = top_component_terms(loadings, 'PC1', top_n=5)
    info = {
        'source_tfidf_table': PCA_INPUT_USED.name,
        'library_used': 'sklearn.decomposition.PCA',
        'number_of_components': comps.shape[0],
        'components_shape': f'{comps.shape[0]} x {comps.shape[1]}',
        'dcm_shape': f'{dcm.shape[0]} x {dcm.shape[1]}',
        'loadings_shape': f'{loadings.shape[0]} x {loadings.shape[1]}',
        'pc0_explained_variance_ratio': explained_index.loc['PC0', 'explained_variance_ratio'],
        'pc1_explained_variance_ratio': explained_index.loc['PC1', 'explained_variance_ratio'],
        'pc2_explained_variance_ratio': explained_index.loc['PC2', 'explained_variance_ratio'],
        'pc3_explained_variance_ratio': explained_index.loc['PC3', 'explained_variance_ratio'],
        'pc0_top_5_positive_terms': ', '.join(pc0_pos.index.tolist()),
        'pc0_top_5_negative_terms': ', '.join(pc0_neg.index.tolist()),
        'pc1_top_5_positive_terms': ', '.join(pc1_pos.index.tolist()),
        'pc1_top_5_negative_terms': ', '.join(pc1_neg.index.tolist()),
    }
    return pd.DataFrame([info])


COMPONENT_SUMMARY = build_component_summary(LOADINGS, EXPLAINED, top_n=5)
PCA_PROJECT_INFO = build_project_info(COMPS, DCM, LOADINGS, EXPLAINED)

print('Final project PCA info:')
print(PCA_PROJECT_INFO.T)
print('\nLeading component summary:')
COMPONENT_SUMMARY.head(4)

Final project PCA info:
                                                                              0
source_tfidf_table                                     tfidf_reduced_l2.parquet
library_used                                          sklearn.decomposition.PCA
number_of_components                                                       3006
components_shape                                                    3006 x 3006
dcm_shape                                                          33726 x 3006
loadings_shape                                                      3006 x 3006
pc0_explained_variance_ratio                                           0.006832
pc1_explained_variance_ratio                                           0.005057
pc2_explained_variance_ratio                                            0.00447
pc3_explained_variance_ratio                                           0.004184
pc0_top_5_positive_terms      social, economic, citizens, development, prote...
pc0_top_5_negati

,component_id,explained_variance,explained_variance_ratio,cumulative_explained_variance_ratio,top_positive_terms,top_negative_terms
0,PC0,0.006741,0.006832,0.006832,"social, economic, citizens, development, prote...","speaker, deputy, ministers, majority, senate"
1,PC1,0.004989,0.005057,0.011889,"judge, high, appeal, courts, judges","ministers, deputies, vote, majority, chamber"
2,PC2,0.004410,0.004470,0.016359,"ministers, courts, local, administrative, judges","speaker, citizen, every, freedom, date"
3,PC3,0.004127,0.004184,0.020542,"ministers, cabinet, deputy, speaker, governor-...","courts, judges, majority, jurisdiction, high"


## PCA Visualizations

These cells produce the main PCA figures for interpretation and the final project. The PCA model may be fit on article- or clause-level bags, but the document-space plots shown here use constitution-level mean PCA scores so each constitution appears once rather than once per article. The section creates:
- document-space scatterplots for `PC0` vs `PC1`
- document-space scatterplots for `PC2` vs `PC3`
- loading-space scatterplots for the same two component pairs
- a scree plot for explained variance

These outputs correspond directly to the two PCA visualization prompts in `FinalProject.ipynb`.


In [81]:
# Visualization 1: country_id plots in the PC0 and PC1 space with region, years, and V-Dem categories.
VIS_PC01_DOC_REGION = vis_pcs(PCA_COUNTRY, 0, 1, 'region_compressed')
VIS_PC01_DOC_REGION.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_YEAR_CREATED = vis_pcs(PCA_COUNTRY, 0, 1, 'year_created')
VIS_PC01_DOC_YEAR_CREATED.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_YEAR = vis_pcs(PCA_COUNTRY, 0, 1, 'year_amended')
VIS_PC01_DOC_YEAR.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_V2X_REGIME = vis_pcs(PCA_COUNTRY, 0, 1, 'v2x_regime_cat')
VIS_PC01_DOC_V2X_REGIME.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_V2X_FREEXP_ALTINF = vis_pcs(PCA_COUNTRY, 0, 1, 'v2x_freexp_altinf_cat')
VIS_PC01_DOC_V2X_FREEXP_ALTINF.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_V2X_RULE = vis_pcs(PCA_COUNTRY, 0, 1, 'v2x_rule_cat')
VIS_PC01_DOC_V2X_RULE.update_layout(title='PCA Visualization 1')
VIS_PC01_LOADINGS = vis_loadings(LOADINGS, 0, 1, top_n=150)
VIS_PC01_LOADINGS.update_layout(title='PCA Visualization 1')
VIS_PC01_DOC_REGION.show()
VIS_PC01_DOC_YEAR_CREATED.show()
VIS_PC01_DOC_YEAR.show()
VIS_PC01_DOC_V2X_REGIME.show()
VIS_PC01_DOC_V2X_FREEXP_ALTINF.show()
VIS_PC01_DOC_V2X_RULE.show()
VIS_PC01_LOADINGS.show()


In [82]:
# Visualization 2: country_id plots in the PC2 and PC3 space with region, years, and V-Dem categories.
VIS_PC23_DOC_REGION = vis_pcs(PCA_COUNTRY, 2, 3, 'region_compressed')
VIS_PC23_DOC_REGION.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_YEAR_CREATED = vis_pcs(PCA_COUNTRY, 2, 3, 'year_created')
VIS_PC23_DOC_YEAR_CREATED.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_YEAR = vis_pcs(PCA_COUNTRY, 2, 3, 'year_amended')
VIS_PC23_DOC_YEAR.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_V2X_REGIME = vis_pcs(PCA_COUNTRY, 2, 3, 'v2x_regime_cat')
VIS_PC23_DOC_V2X_REGIME.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_V2X_FREEXP_ALTINF = vis_pcs(PCA_COUNTRY, 2, 3, 'v2x_freexp_altinf_cat')
VIS_PC23_DOC_V2X_FREEXP_ALTINF.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_V2X_RULE = vis_pcs(PCA_COUNTRY, 2, 3, 'v2x_rule_cat')
VIS_PC23_DOC_V2X_RULE.update_layout(title='PCA Visualization 2')
VIS_PC23_LOADINGS = vis_loadings(LOADINGS, 2, 3, top_n=150)
VIS_PC23_LOADINGS.update_layout(title='PCA Visualization 2')
VIS_PC23_DOC_REGION.show()
VIS_PC23_DOC_YEAR_CREATED.show()
VIS_PC23_DOC_YEAR.show()
VIS_PC23_DOC_V2X_REGIME.show()
VIS_PC23_DOC_V2X_FREEXP_ALTINF.show()
VIS_PC23_DOC_V2X_RULE.show()
VIS_PC23_LOADINGS.show()


In [83]:
# Optional scree plot for discussing explained variance in the write-up.
SCREE = vis_scree(EXPLAINED, top_n=10)
SCREE.show()


## Save PCA Tables

The final cell writes the four reusable PCA tables to Parquet so the final project notebook can reference stable saved outputs rather than transient in-memory objects.


In [84]:
# Save the core PCA outputs for the final project notebook and write-up.
COMPS.to_parquet(COMPS_PARQUET)
DCM.to_parquet(DCM_PARQUET)
LOADINGS.to_parquet(LOADINGS_PARQUET)
EXPLAINED.to_parquet(EXPLAINED_VARIANCE_PARQUET, index=False)

print(f'Saved COMPS to: {COMPS_PARQUET.resolve()}')
print(f'Saved DCM to: {DCM_PARQUET.resolve()}')
print(f'Saved LOADINGS to: {LOADINGS_PARQUET.resolve()}')
print(f'Saved explained variance table to: {EXPLAINED_VARIANCE_PARQUET.resolve()}')
print('PCA visualizations are displayed directly in notebook cells.')
print('PCA summary details are displayed in the notebook cells above and have been written into FinalProject.ipynb.')


# Save interactive Plotly HTML files for embedding in other notebooks.
VIS_PC01_LOADINGS.write_html("pca_pc01_loadings.html", include_plotlyjs=True)
VIS_PC01_DOC_REGION.write_html("pca_pc01_doc_region.html", include_plotlyjs=True)
VIS_PC23_LOADINGS.write_html("pca_pc23_loadings.html", include_plotlyjs=True)
VIS_PC23_DOC_REGION.write_html("pca_pc23_doc_region.html", include_plotlyjs=True)
print(f'Saved interactive loadings figure to: {(Path("pca_pc01_loadings.html")).resolve()}')
print(f'Saved interactive region figure to: {(Path("pca_pc01_doc_region.html")).resolve()}')
print(f'Saved interactive loadings figure to: {(Path("pca_pc23_loadings.html")).resolve()}')
print(f'Saved interactive region figure to: {(Path("pca_pc23_doc_region.html")).resolve()}')

Saved COMPS to: C:\Users\garre\school\spring_2026\ds_5001\pca_comps.parquet
Saved DCM to: C:\Users\garre\school\spring_2026\ds_5001\pca_dcm.parquet
Saved LOADINGS to: C:\Users\garre\school\spring_2026\ds_5001\pca_loadings.parquet
Saved explained variance table to: C:\Users\garre\school\spring_2026\ds_5001\pca_explained_variance.parquet
PCA visualizations are displayed directly in notebook cells.
PCA summary details are displayed in the notebook cells above and have been written into FinalProject.ipynb.
Saved interactive loadings figure to: C:\Users\garre\school\spring_2026\ds_5001\pca_pc01_loadings.html
Saved interactive region figure to: C:\Users\garre\school\spring_2026\ds_5001\pca_pc01_doc_region.html
Saved interactive loadings figure to: C:\Users\garre\school\spring_2026\ds_5001\pca_pc23_loadings.html
Saved interactive region figure to: C:\Users\garre\school\spring_2026\ds_5001\pca_pc23_doc_region.html
